# Chapter 29: Dynamic Programming I: Memoization and Tables

The programs from Chapter 29 of *Data Structures and Algorithms in Python, Hands-On* by Aadhavan Muthurengan, in the book's order. Each code cell is one program; the same program is in this folder as a `.py` file, and its expected output is in the `.out` file with the same name. Benchmark timings differ on every computer and every run.

Run the cells from top to bottom (**Run > Run All Cells**). The programs import each other's files, so keep this notebook in its chapter folder.

## Climbing Stairs

`dp_basics.py`

In [1]:
"""Dynamic programming: solve each smaller problem once, store the
answer, and build bigger answers from smaller ones."""

import math
from functools import cache


def climb_ways(n):
    """Ways to climb n stairs taking 1 or 2 at a time. The last step was
    a 1 or a 2, so ways(n) = ways(n - 1) + ways(n - 2)."""
    a, b = 1, 1  # ways(0), ways(1)
    for _ in range(n - 1):
        a, b = b, a + b
    return b if n else 1


def rob(houses):
    """The most money from houses in a row, never two neighbors. Each
    house: skip it (best so far) or take it (best two back + it)."""
    two_back = one_back = 0
    for money in houses:
        two_back, one_back = one_back, max(one_back, two_back + money)
    return one_back


def fewest_coins_memo(coins, amount):
    """Top-down: recursion plus a cache."""

    @cache
    def best(a):
        if a == 0:
            return 0
        options = [best(a - c) + 1 for c in coins if c <= a]
        return min(options, default=math.inf)

    answer = best(amount)
    return -1 if answer == math.inf else answer


def fewest_coins(coins, amount):
    """Bottom-up: fill a table from 0 up to amount. table[a] is the
    fewest coins that make a (inf if none)."""
    table = [0] + [math.inf] * amount
    for a in range(1, amount + 1):
        for c in coins:
            if c <= a and table[a - c] + 1 < table[a]:
                table[a] = table[a - c] + 1
    return -1 if table[amount] == math.inf else table[amount]


def coin_ways(coins, amount):
    """How many combinations of coins make amount (order doesn't
    matter). Looping over coins on the outside counts each combination
    once."""
    ways = [1] + [0] * amount
    for c in coins:
        for a in range(c, amount + 1):
            ways[a] += ways[a - c]
    return ways[amount]


def min_path_sum(grid):
    """The cheapest path from top-left to bottom-right moving only right
    or down: each cell's best is its cost plus the better of the cells
    above and to the left."""
    rows, cols = len(grid), len(grid[0])
    best = [[0] * cols for _ in range(rows)]
    for r in range(rows):
        for c in range(cols):
            above = best[r - 1][c] if r else math.inf
            left = best[r][c - 1] if c else math.inf
            start = 0 if r == c == 0 else min(above, left)
            best[r][c] = grid[r][c] + start
    return best[-1][-1]

## Fewest Coins

`coin_table.py`

In [2]:
"""Watch the bottom-up table fill: fewest coins for each amount 0..11,
with coins 1, 2 and 5."""

import math

coins, amount = [1, 2, 5], 11
table = [0] + [math.inf] * amount
for a in range(1, amount + 1):
    for c in coins:
        if c <= a and table[a - c] + 1 < table[a]:
            table[a] = table[a - c] + 1
print("amount: " + " ".join(f"{a:>2}" for a in range(amount + 1)))
print("coins:  " + " ".join(f"{t:>2}" for t in table))

amount:  0  1  2  3  4  5  6  7  8  9 10 11
coins:   0  1  1  2  2  1  2  2  3  3  2  3


## Paths Through a Grid

`dp_demo.py`

In [3]:
"""The DP solutions on small examples."""

from dp_basics import (
    climb_ways,
    coin_ways,
    fewest_coins,
    min_path_sum,
    rob,
)

print([climb_ways(n) for n in range(1, 11)])
print(rob([2, 7, 9, 3, 1]), rob([5, 1, 1, 5]))
print(fewest_coins([1, 5, 10, 25], 63), fewest_coins([2], 3))
print(coin_ways([1, 2, 5], 5))
print(min_path_sum([[1, 3, 1], [1, 5, 1], [4, 2, 1]]))

[1, 2, 3, 5, 8, 13, 21, 34, 55, 89]
12 10
6 -1
4
7


## Timing

`bench_coins.py`

In [4]:
"""Fewest coins: plain recursion, recursion with a cache, and a table."""

import math
import sys

import bench
from dp_basics import fewest_coins, fewest_coins_memo

sys.setrecursionlimit(10_000)
COINS = [1, 5, 10, 25]


def plain(amount):
    if amount == 0:
        return 0
    return min(
        (plain(amount - c) + 1 for c in COINS if c <= amount),
        default=math.inf,
    )


sizes = [20, 25, 30, 35]
results = {
    "plain recursion": bench.doubling(plain, lambda n: (n,), sizes),
    "memoized": bench.doubling(
        fewest_coins_memo, lambda n: (COINS, n), sizes
    ),
    "table": bench.doubling(fewest_coins, lambda n: (COINS, n), sizes),
}
bench.table(results)

         n  plain recursion        memoized           table
        20           172 us         8.03 us         2.96 us
        25      790 us x4.6    10.2 us x1.3    3.71 us x1.3
        30     3.68 ms x4.7    11.3 us x1.1    4.66 us x1.3
        35       17 ms x4.6    12.7 us x1.1    5.49 us x1.2
